In [2]:
import warnings

warnings.filterwarnings("ignore")

import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn.metrics as metrics
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder

pd.options.display.float_format = "{:.2f}".format

In [3]:
df = pd.read_csv("data/CVD_cleaned.csv")

In [4]:
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,Poor,Within the past 2 years,No,No,No,No,No,No,Yes,Female,70-74,150.00,32.66,14.54,Yes,0.00,30.00,16.00,12.00
1,Very Good,Within the past year,No,Yes,No,No,No,Yes,No,Female,70-74,165.00,77.11,28.29,No,0.00,30.00,0.00,4.00
2,Very Good,Within the past year,Yes,No,No,No,No,Yes,No,Female,60-64,163.00,88.45,33.47,No,4.00,12.00,3.00,16.00
3,Poor,Within the past year,Yes,Yes,No,No,No,Yes,No,Male,75-79,180.00,93.44,28.73,No,0.00,30.00,30.00,8.00
4,Good,Within the past year,No,No,No,No,No,No,No,Male,80+,191.00,88.45,24.37,Yes,0.00,8.00,4.00,0.00


In [5]:
df.drop_duplicates()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,Poor,Within the past 2 years,No,No,No,No,No,No,Yes,Female,70-74,150.00,32.66,14.54,Yes,0.00,30.00,16.00,12.00
1,Very Good,Within the past year,No,Yes,No,No,No,Yes,No,Female,70-74,165.00,77.11,28.29,No,0.00,30.00,0.00,4.00
2,Very Good,Within the past year,Yes,No,No,No,No,Yes,No,Female,60-64,163.00,88.45,33.47,No,4.00,12.00,3.00,16.00
3,Poor,Within the past year,Yes,Yes,No,No,No,Yes,No,Male,75-79,180.00,93.44,28.73,No,0.00,30.00,30.00,8.00
4,Good,Within the past year,No,No,No,No,No,No,No,Male,80+,191.00,88.45,24.37,Yes,0.00,8.00,4.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308849,Very Good,Within the past year,Yes,No,No,No,No,No,No,Male,25-29,168.00,81.65,29.05,No,4.00,30.00,8.00,0.00
308850,Fair,Within the past 5 years,Yes,No,No,No,No,Yes,No,Male,65-69,180.00,69.85,21.48,No,8.00,15.00,60.00,4.00
308851,Very Good,5 or more years ago,Yes,No,No,No,Yes,"Yes, but female told only during pregnancy",No,Female,30-34,157.00,61.23,24.69,Yes,4.00,40.00,8.00,4.00
308852,Very Good,Within the past year,Yes,No,No,No,No,No,No,Male,65-69,183.00,79.38,23.73,No,3.00,30.00,12.00,0.00


In [6]:
df.isnull().sum()

General_Health                  0
Checkup                         0
Exercise                        0
Heart_Disease                   0
Skin_Cancer                     0
Other_Cancer                    0
Depression                      0
Diabetes                        0
Arthritis                       0
Sex                             0
Age_Category                    0
Height_(cm)                     0
Weight_(kg)                     0
BMI                             0
Smoking_History                 0
Alcohol_Consumption             0
Fruit_Consumption               0
Green_Vegetables_Consumption    0
FriedPotato_Consumption         0
dtype: int64

In [7]:
for i in df.columns:
    dis = len(df[i].unique())
    print(f"{i} - {dis}")

General_Health - 5
Checkup - 5
Exercise - 2
Heart_Disease - 2
Skin_Cancer - 2
Other_Cancer - 2
Depression - 2
Diabetes - 4
Arthritis - 2
Sex - 2
Age_Category - 13
Height_(cm) - 99
Weight_(kg) - 525
BMI - 3654
Smoking_History - 2
Alcohol_Consumption - 31
Fruit_Consumption - 77
Green_Vegetables_Consumption - 75
FriedPotato_Consumption - 69


In [8]:
df["Checkup"].value_counts()

Checkup
Within the past year       239371
Within the past 2 years     37213
Within the past 5 years     17442
5 or more years ago         13421
Never                        1407
Name: count, dtype: int64

In [9]:
df["Diabetes"].value_counts()

Diabetes
No                                            259141
Yes                                            40171
No, pre-diabetes or borderline diabetes         6896
Yes, but female told only during pregnancy      2646
Name: count, dtype: int64

In [10]:
enc = OrdinalEncoder()
df["Exercise"] = enc.fit_transform(df[["Exercise"]])
df["Heart_Disease"] = enc.fit_transform(df[["Heart_Disease"]])
df["Skin_Cancer"] = enc.fit_transform(df[["Skin_Cancer"]])
df["Other_Cancer"] = enc.fit_transform(df[["Other_Cancer"]])
df["Depression"] = enc.fit_transform(df[["Depression"]])
df["Sex"] = enc.fit_transform(df[["Sex"]])
df["Arthritis"] = enc.fit_transform(df[["Arthritis"]])
df["Smoking_History"] = enc.fit_transform(df[["Smoking_History"]])
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,Poor,Within the past 2 years,0.00,0.00,0.00,0.00,0.00,No,1.00,0.00,70-74,150.00,32.66,14.54,1.00,0.00,30.00,16.00,12.00
1,Very Good,Within the past year,0.00,1.00,0.00,0.00,0.00,Yes,0.00,0.00,70-74,165.00,77.11,28.29,0.00,0.00,30.00,0.00,4.00
2,Very Good,Within the past year,1.00,0.00,0.00,0.00,0.00,Yes,0.00,0.00,60-64,163.00,88.45,33.47,0.00,4.00,12.00,3.00,16.00
3,Poor,Within the past year,1.00,1.00,0.00,0.00,0.00,Yes,0.00,1.00,75-79,180.00,93.44,28.73,0.00,0.00,30.00,30.00,8.00
4,Good,Within the past year,0.00,0.00,0.00,0.00,0.00,No,0.00,1.00,80+,191.00,88.45,24.37,1.00,0.00,8.00,4.00,0.00


In [11]:
rank = ["Poor", "Fair", "Good", "Very Good", "Excellent"]
oe = OrdinalEncoder(categories=[rank])
df["General_Health"] = oe.fit_transform(df[["General_Health"]])
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,0.00,Within the past 2 years,0.00,0.00,0.00,0.00,0.00,No,1.00,0.00,70-74,150.00,32.66,14.54,1.00,0.00,30.00,16.00,12.00
1,3.00,Within the past year,0.00,1.00,0.00,0.00,0.00,Yes,0.00,0.00,70-74,165.00,77.11,28.29,0.00,0.00,30.00,0.00,4.00
2,3.00,Within the past year,1.00,0.00,0.00,0.00,0.00,Yes,0.00,0.00,60-64,163.00,88.45,33.47,0.00,4.00,12.00,3.00,16.00
3,0.00,Within the past year,1.00,1.00,0.00,0.00,0.00,Yes,0.00,1.00,75-79,180.00,93.44,28.73,0.00,0.00,30.00,30.00,8.00
4,2.00,Within the past year,0.00,0.00,0.00,0.00,0.00,No,0.00,1.00,80+,191.00,88.45,24.37,1.00,0.00,8.00,4.00,0.00


In [12]:
rank1 = [
    "No",
    "No, pre-diabetes or borderline diabetes",
    "Yes, but female told only during pregnancy",
    "Yes",
]
oe1 = OrdinalEncoder(categories=[rank1])
df["Diabetes"] = oe1.fit_transform(df[["Diabetes"]])
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,0.00,Within the past 2 years,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,70-74,150.00,32.66,14.54,1.00,0.00,30.00,16.00,12.00
1,3.00,Within the past year,0.00,1.00,0.00,0.00,0.00,3.00,0.00,0.00,70-74,165.00,77.11,28.29,0.00,0.00,30.00,0.00,4.00
2,3.00,Within the past year,1.00,0.00,0.00,0.00,0.00,3.00,0.00,0.00,60-64,163.00,88.45,33.47,0.00,4.00,12.00,3.00,16.00
3,0.00,Within the past year,1.00,1.00,0.00,0.00,0.00,3.00,0.00,1.00,75-79,180.00,93.44,28.73,0.00,0.00,30.00,30.00,8.00
4,2.00,Within the past year,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,80+,191.00,88.45,24.37,1.00,0.00,8.00,4.00,0.00


In [13]:
rank2 = [
    "Within the past year",
    "Within the past 2 years",
    "Within the past 5 years",
    "5 or more years ago",
    "Never",
]
oe2 = OrdinalEncoder(categories=[rank2])
df["Checkup"] = oe2.fit_transform(df[["Checkup"]])
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,70-74,150.00,32.66,14.54,1.00,0.00,30.00,16.00,12.00
1,3.00,0.00,0.00,1.00,0.00,0.00,0.00,3.00,0.00,0.00,70-74,165.00,77.11,28.29,0.00,0.00,30.00,0.00,4.00
2,3.00,0.00,1.00,0.00,0.00,0.00,0.00,3.00,0.00,0.00,60-64,163.00,88.45,33.47,0.00,4.00,12.00,3.00,16.00
3,0.00,0.00,1.00,1.00,0.00,0.00,0.00,3.00,0.00,1.00,75-79,180.00,93.44,28.73,0.00,0.00,30.00,30.00,8.00
4,2.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,80+,191.00,88.45,24.37,1.00,0.00,8.00,4.00,0.00


In [14]:
df["Age_Category"] = df["Age_Category"].str[:2].astype(int)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308854 entries, 0 to 308853
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   General_Health                308854 non-null  float64
 1   Checkup                       308854 non-null  float64
 2   Exercise                      308854 non-null  float64
 3   Heart_Disease                 308854 non-null  float64
 4   Skin_Cancer                   308854 non-null  float64
 5   Other_Cancer                  308854 non-null  float64
 6   Depression                    308854 non-null  float64
 7   Diabetes                      308854 non-null  float64
 8   Arthritis                     308854 non-null  float64
 9   Sex                           308854 non-null  float64
 10  Age_Category                  308854 non-null  int64  
 11  Height_(cm)                   308854 non-null  float64
 12  Weight_(kg)                   308854 non-nul

In [16]:
df["Heart_Disease"].value_counts()

Heart_Disease
0.00    283883
1.00     24971
Name: count, dtype: int64

In [ ]:
sns.distplot(df["Age_Category"], bins=10)

In [ ]:
X = df.drop(["Heart_Disease"], axis=1)
y = df["Heart_Disease"]

In [ ]:
sm = SMOTE(sampling_strategy="minority")
X, y = sm.fit_resample(X, y)
y.value_counts()

Heart_Disease
0.00    283883
1.00    283883
Name: count, dtype: int64

In [ ]:
scalar = MinMaxScaler()
X1 = scalar.fit_transform(X)
X1.shape

(567766, 18)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, stratify=y)
X_train1 = X_train[:80000]
y_train1 = y_train[:80000]

In [ ]:
rf = RandomForestClassifier(
    min_samples_leaf=4, max_depth=10, min_samples_split=10, n_estimators=100
)

In [ ]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,10
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y1_pred = rf.predict(X_test)
cm = metrics.confusion_matrix(y_test, y1_pred)
print(classification_report(y_test, y1_pred))

              precision    recall  f1-score   support

         0.0       0.92      0.96      0.94     56777
         1.0       0.95      0.92      0.93     56777

    accuracy                           0.94    113554
   macro avg       0.94      0.94      0.94    113554
weighted avg       0.94      0.94      0.94    113554



In [ ]:
joblib.dump(rf, "artifacts/heart_300k_rf_model.pkl")

schema = {
    "features": list(X.columns),
    "dtypes": {col: str(dtype) for col, dtype in X.dtypes.items()},
    "target": "Heart_Disease",
}

with open("artifacts/heart_300k_rf_schema.json", "w") as f:
    json.dump(schema, f, indent=4)